# 02 — Data Understanding: Exploração Inicial dos Dados

**Projeto:** PoD Bank — Credit Score Model  
**Fase CRISP-DM:** 2 — Data Understanding  
**Objetivo:** Entender a estrutura, qualidade e relacionamentos de todos os datasets do Home Credit.

---

## Esquema de Relacionamento entre Tabelas

```
application_train / application_test   (tabela central)
        │
        │ SK_ID_CURR
        ├──────────────────► bureau.csv
        │                         │ SK_ID_BUREAU
        │                         └──────────► bureau_balance.csv
        │
        │ SK_ID_CURR
        ├──────────────────► previous_application.csv
        │
        │ SK_ID_CURR
        ├──────────────────► POS_CASH_balance.csv
        │
        │ SK_ID_CURR
        ├──────────────────► credit_card_balance.csv
        │
        │ SK_ID_CURR
        └──────────────────► installments_payments.csv
```

| Tabela | Chave para application | Granularidade |
|---|---|---|
| `bureau.csv` | `SK_ID_CURR` | 1 linha por crédito externo |
| `bureau_balance.csv` | `SK_ID_BUREAU` (via bureau) | 1 linha por mês por crédito |
| `previous_application.csv` | `SK_ID_CURR` | 1 linha por aplicação anterior |
| `POS_CASH_balance.csv` | `SK_ID_CURR` | 1 linha por mês por contrato |
| `credit_card_balance.csv` | `SK_ID_CURR` | 1 linha por mês por cartão |
| `installments_payments.csv` | `SK_ID_CURR` | 1 linha por parcela paga/devida |

---
## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

DATA_PATH = '../../data/raw/'

In [ ]:
def dataset_overview(df: pd.DataFrame, name: str, key_col: str = None) -> None:
    """Imprime shape, memory, tipos de variáveis, nulos e head."""
    print(f'{'='*60}')
    print(f'  {name}')
    print(f'{'='*60}')

    # Shape e memória
    mem_mb = df.memory_usage(deep=True).sum() / 1024**2
    print(f'\n Shape:        {df.shape[0]:,} linhas × {df.shape[1]} colunas')
    print(f' Memória:      {mem_mb:.1f} MB')

    # Tipos de variáveis
    type_counts = df.dtypes.value_counts()
    print(f'\n Tipos de variáveis:')
    for dtype, count in type_counts.items():
        print(f'   {str(dtype):<12} {count:>4} colunas')

    # Nulos — top 10
    null_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
    null_pct = null_pct[null_pct > 0].head(10)
    if len(null_pct) > 0:
        print(f'\n % Nulos — Top {len(null_pct)}:')
        for col, pct in null_pct.items():
            bar = '█' * int(pct / 5)
            print(f'   {col:<45} {pct:>6.1f}%  {bar}')
    else:
        print('\n Sem valores nulos.')

    # Chave de relacionamento
    if key_col and key_col in df.columns:
        n_unique = df[key_col].nunique()
        print(f'\n Chave [{key_col}]:  {n_unique:,} valores únicos  '
              f'(media {len(df)/n_unique:.1f} linhas por chave)')

    print(f'\n Head(3):')
    display(df.head(3))
    print()

---
## 1. application_train.csv

Tabela central do projeto. Cada linha representa uma aplicação de crédito.  
**TARGET**: `1` = inadimplente (mau pagador), `0` = adimplente (bom pagador).

In [ ]:
app_train = pd.read_csv(DATA_PATH + 'application_train.csv')
dataset_overview(app_train, 'application_train.csv', key_col='SK_ID_CURR')

---
## 2. application_test.csv

Mesmo esquema que application_train, sem a coluna TARGET.  
Usado para scoring final — sem rótulo de inadimplência.

In [ ]:
app_test = pd.read_csv(DATA_PATH + 'application_test.csv')
dataset_overview(app_test, 'application_test.csv', key_col='SK_ID_CURR')

# Comparação de cobertura
overlap = len(set(app_train['SK_ID_CURR']) & set(app_test['SK_ID_CURR']))
print(f' Overlap de SK_ID_CURR entre train e test: {overlap} IDs')

---
## 3. bureau.csv

Histórico de crédito em outras instituições financeiras (fornecido pelo bureau de crédito).  
**Chave para application:** `SK_ID_CURR` — um cliente pode ter múltiplos créditos externos.

In [ ]:
bureau = pd.read_csv(DATA_PATH + 'bureau.csv')
dataset_overview(bureau, 'bureau.csv', key_col='SK_ID_CURR')

# Cobertura em relação ao train
coverage = bureau['SK_ID_CURR'].isin(app_train['SK_ID_CURR']).sum()
pct = coverage / len(bureau) * 100
print(f' Linhas com SK_ID_CURR presente no train: {coverage:,} ({pct:.1f}%)')

---
## 4. bureau_balance.csv

Saldo mensal de cada crédito registrado no bureau.  
**Chave para bureau:** `SK_ID_BUREAU` — um crédito bureau pode ter múltiplos meses de histórico.

In [ ]:
bureau_bal = pd.read_csv(DATA_PATH + 'bureau_balance.csv')
dataset_overview(bureau_bal, 'bureau_balance.csv', key_col='SK_ID_BUREAU')

# Cobertura via bureau
ids_bureau = set(bureau['SK_ID_BUREAU'])
coverage = bureau_bal['SK_ID_BUREAU'].isin(ids_bureau).sum()
pct = coverage / len(bureau_bal) * 100
print(f' Linhas com SK_ID_BUREAU presente em bureau.csv: {coverage:,} ({pct:.1f}%)')

print(f'\n STATUS (status do contrato):')
display(bureau_bal['STATUS'].value_counts().to_frame('count').assign(
    pct=lambda x: (x['count'] / len(bureau_bal) * 100).round(2)
))

---
## 5. previous_application.csv

Aplicações de crédito anteriores do mesmo cliente na PoD Bank.  
**Chave para application:** `SK_ID_CURR` — um cliente pode ter múltiplas aplicações anteriores.

In [ ]:
prev_app = pd.read_csv(DATA_PATH + 'previous_application.csv')
dataset_overview(prev_app, 'previous_application.csv', key_col='SK_ID_CURR')

coverage = prev_app['SK_ID_CURR'].isin(app_train['SK_ID_CURR']).sum()
pct = coverage / len(prev_app) * 100
print(f' Linhas com SK_ID_CURR presente no train: {coverage:,} ({pct:.1f}%)')

---
## 6. POS_CASH_balance.csv

Saldo mensal de empréstimos POS (ponto de venda) e empréstimos em dinheiro anteriores.  
**Chave para application:** `SK_ID_CURR`.

In [ ]:
pos_cash = pd.read_csv(DATA_PATH + 'POS_CASH_balance.csv')
dataset_overview(pos_cash, 'POS_CASH_balance.csv', key_col='SK_ID_CURR')

coverage = pos_cash['SK_ID_CURR'].isin(app_train['SK_ID_CURR']).sum()
pct = coverage / len(pos_cash) * 100
print(f' Linhas com SK_ID_CURR presente no train: {coverage:,} ({pct:.1f}%)')

---
## 7. credit_card_balance.csv

Extratos mensais de cartões de crédito anteriores emitidos pela PoD Bank.  
**Chave para application:** `SK_ID_CURR`.

In [ ]:
cc_bal = pd.read_csv(DATA_PATH + 'credit_card_balance.csv')
dataset_overview(cc_bal, 'credit_card_balance.csv', key_col='SK_ID_CURR')

coverage = cc_bal['SK_ID_CURR'].isin(app_train['SK_ID_CURR']).sum()
pct = coverage / len(cc_bal) * 100
print(f' Linhas com SK_ID_CURR presente no train: {coverage:,} ({pct:.1f}%)')

---
## 8. installments_payments.csv

Histórico de pagamento de parcelas de créditos anteriores (data e valor pago vs. devido).  
**Chave para application:** `SK_ID_CURR`.

In [ ]:
install = pd.read_csv(DATA_PATH + 'installments_payments.csv')
dataset_overview(install, 'installments_payments.csv', key_col='SK_ID_CURR')

coverage = install['SK_ID_CURR'].isin(app_train['SK_ID_CURR']).sum()
pct = coverage / len(install) * 100
print(f' Linhas com SK_ID_CURR presente no train: {coverage:,} ({pct:.1f}%)')

---
## 9. Análise do TARGET (application_train)

O TARGET é a variável dependente do modelo.  
`0` = adimplente | `1` = inadimplente (evento de crédito)

In [ ]:
target_counts = app_train['TARGET'].value_counts()
target_pct = app_train['TARGET'].value_counts(normalize=True) * 100

print('Distribuição do TARGET')
print(f'  Adimplentes  (0): {target_counts[0]:>8,}  ({target_pct[0]:.2f}%)')
print(f'  Inadimplentes(1): {target_counts[1]:>8,}  ({target_pct[1]:.2f}%)')
print(f'  Razão de desbalanceamento: {target_counts[0]/target_counts[1]:.1f}:1')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Barras de contagem
colors = ['#2ecc71', '#e74c3c']
axes[0].bar(['Adimplente (0)', 'Inadimplente (1)'], target_counts.values, color=colors, edgecolor='white', linewidth=1.5)
axes[0].set_title('Contagem por Classe (TARGET)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Quantidade de aplicações')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
for i, (bar, v) in enumerate(zip(axes[0].patches, target_counts.values)):
    axes[0].text(bar.get_x() + bar.get_width()/2, v + 500, f'{v:,}', ha='center', fontsize=11)

# Pizza
axes[1].pie(
    target_counts.values,
    labels=[f'Adimplente\n{target_pct[0]:.1f}%', f'Inadimplente\n{target_pct[1]:.1f}%'],
    colors=colors,
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2},
    textprops={'fontsize': 11}
)
axes[1].set_title('Proporção das Classes (TARGET)', fontsize=13, fontweight='bold')

plt.suptitle('Desbalanceamento de Classes — TARGET', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../../reports/figures/target_distribution.png', bbox_inches='tight')
plt.show()

---
## 10. Correlações com o TARGET

Análise das variáveis numéricas com maior correlação absoluta com o TARGET.

In [ ]:
# Correlações de Pearson das numéricas com TARGET
numeric_cols = app_train.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in ['SK_ID_CURR', 'TARGET']]

corr_target = (
    app_train[numeric_cols + ['TARGET']]
    .corr()['TARGET']
    .drop('TARGET')
    .abs()
    .sort_values(ascending=False)
)

top_n = 20
top_corr = corr_target.head(top_n)

print(f'Top {top_n} variáveis com maior correlação absoluta com TARGET:')
display(top_corr.to_frame('|corr|').assign(
    corr_signed=lambda _: app_train[top_corr.index].corrwith(app_train['TARGET']).round(4)
).round(4))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

signed_corr = app_train[top_corr.index].corrwith(app_train['TARGET']).sort_values()
bar_colors = ['#e74c3c' if v > 0 else '#3498db' for v in signed_corr.values]

ax.barh(signed_corr.index, signed_corr.values, color=bar_colors, edgecolor='white', linewidth=0.8)
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('Correlação de Pearson com TARGET')
ax.set_title(f'Top {top_n} Correlações com TARGET (vermelho = positiva, azul = negativa)',
             fontsize=12, fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.3f'))

plt.tight_layout()
plt.savefig('../../reports/figures/target_correlations.png', bbox_inches='tight')
plt.show()

In [ ]:
# Distribuição das top 3 features por TARGET
top3 = corr_target.head(3).index.tolist()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, top3):
    for target_val, label, color in [(0, 'Adimplente', '#2ecc71'), (1, 'Inadimplente', '#e74c3c')]:
        subset = app_train.loc[app_train['TARGET'] == target_val, col].dropna()
        ax.hist(subset, bins=40, alpha=0.6, label=label, color=color, density=True)
    ax.set_title(col, fontsize=10, fontweight='bold')
    ax.legend(fontsize=8)
    ax.set_xlabel('')

plt.suptitle('Distribuição das Top 3 Features por TARGET', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../../reports/figures/top3_features_by_target.png', bbox_inches='tight')
plt.show()

---
## 11. Resumo dos Datasets

In [ ]:
datasets = {
    'application_train': app_train,
    'application_test':  app_test,
    'bureau':            bureau,
    'bureau_balance':    bureau_bal,
    'previous_application': prev_app,
    'POS_CASH_balance':  pos_cash,
    'credit_card_balance': cc_bal,
    'installments_payments': install,
}

summary_rows = []
for name, df in datasets.items():
    null_pct_mean = df.isnull().mean().mean() * 100
    mem_mb = df.memory_usage(deep=True).sum() / 1024**2
    n_num = df.select_dtypes(include=np.number).shape[1]
    n_cat = df.select_dtypes(include='object').shape[1]
    summary_rows.append({
        'Dataset': name,
        'Linhas': f'{df.shape[0]:,}',
        'Colunas': df.shape[1],
        'Numéricas': n_num,
        'Categóricas': n_cat,
        '% Nulos (média)': f'{null_pct_mean:.1f}%',
        'Memória (MB)': f'{mem_mb:.0f}',
    })

display(pd.DataFrame(summary_rows).set_index('Dataset'))

---
## 12. Conclusões e Próximos Passos

### Principais Observações

**Desbalanceamento de Classes**  
O TARGET é fortemente desbalanceado (~8-9% de inadimplentes). Estratégias a considerar na modelagem: class_weight, SMOTE, threshold calibration.

**Qualidade dos Dados**  
- `application_train/test` possuem colunas com alto percentual de nulos (EXT_SOURCE_*, OCCUPATION_TYPE, etc.) que precisam de estratégia de imputação cuidadosa.
- `bureau.csv` tem variáveis de valor monetário com nulos substanciais.
- `previous_application.csv` contém valores `365243` em colunas de dias — código especial para dados ausentes.

**Granularidade das Tabelas Secundárias**  
Todas as tabelas secundárias são `1:N` em relação à tabela central. Para usá-las no modelo, será necessário agregar por `SK_ID_CURR` (média, soma, contagem, min, max, desvio padrão).

**Sinais Promissores**  
As variáveis `EXT_SOURCE_1/2/3` (scores externos de bureau) aparecem com as mais altas correlações com o TARGET — prioridade na feature engineering.

---

### Próximos Passos (Fase 3 — Data Preparation)

- [ ] Tratar valores especiais (`365243`, `XNA`, `XAP`) nas tabelas secundárias
- [ ] Definir estratégia de imputação por tipo de variável (mediana para numéricas, moda/categoria 'Unknown' para categóricas)
- [ ] Criar features agregadas de bureau (média de dias de atraso, total de créditos ativos)
- [ ] Criar features agregadas de installments (taxa de pagamento no prazo, atraso médio)
- [ ] Criar features agregadas de previous_application (taxa de aprovação histórica, valor médio solicitado)
- [ ] Encodar variáveis categóricas (Label Encoding para ordinais, Target Encoding ou WoE para nominais)
- [ ] Validar ausência de data leakage antes de treinar qualquer modelo